<a href="https://colab.research.google.com/github/junseok-jay/AI_lab/blob/main/pipeline/Llama_split_test_gc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 시작

In [1]:
import torch, transformers
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("cuda:", torch.cuda.is_available(), "gpus:", torch.cuda.device_count())

torch: 2.10.0+cu128
transformers: 5.0.0
cuda: True gpus: 1


In [2]:
import os
import torch
import torch.nn as nn

def stabilize_for_capture(model):
    model.eval()
    if hasattr(model, "config"):
        model.config.use_cache = False
        model.config.return_dict = False
        if hasattr(model.config, "output_attentions"):
            model.config.output_attentions = False
        if hasattr(model.config, "output_hidden_states"):
            model.config.output_hidden_states = False
        if hasattr(model.config, "attn_implementation"):
            # flash/sdpa가 export/unflatten에서 문제를 일으키는 경우가 많아 eager 고정
            model.config.attn_implementation = "eager"
    if hasattr(model, "gradient_checkpointing_disable"):
        try:
            model.gradient_checkpointing_disable()
        except Exception:
            pass
    return model


class SimpleLlamaForPipeline(nn.Module):
    """
    - 루트를 이 wrapper로 고정하면 split_spec은 항상 "model.layers.N"로 고정 가능
    - logits 텐서만 반환 (dict/tuple 복잡성 제거)
    - use_cache/return_dict 등 옵션 고정
    """
    def __init__(self, base_causal_lm: nn.Module):
        super().__init__()
        self.base = base_causal_lm          # LlamaForCausalLM
        self.model = base_causal_lm.model   # LlamaModel  <-- split 경로의 루트
        self.lm_head = base_causal_lm.lm_head

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor | None = None):
        out = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
            return_dict=False,
        )
        hidden = out[0]              # [B,S,H]
        logits = self.lm_head(hidden)  # [B,S,V]
        return logits

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
DTYPE  = torch.float16 if DEVICE.type == "cuda" else torch.float32

# TODO: 너 환경에 맞게 바꾸기
MODEL_ID_OR_PATH = "meta-llama/Llama-3.2-3B-Instruct"   # 또는 "meta-llama/Llama-3.2-3B-Instruct"
# TOKENIZER_ID_OR_PATH = MODEL_ID_OR_PATH  # 보통 동일
TOKENIZER_ID_OR_PATH = MODEL_ID_OR_PATH

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID_OR_PATH, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID_OR_PATH,
    torch_dtype=DTYPE,
    device_map=None,
)
base = stabilize_for_capture(base).to(DEVICE)

root = SimpleLlamaForPipeline(base).to(DEVICE).eval()

print("root type:", type(root))
print("layers:", len(root.model.layers))
print("vocab:", tokenizer.vocab_size)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

root type: <class '__main__.SimpleLlamaForPipeline'>
layers: 28
vocab: 128000


In [4]:
MB_SIZE = 1
SEQ_LEN = 256   # 처음엔 128~256으로 (2048은 성공 후)

mb_kwargs = {
    "input_ids": torch.randint(0, tokenizer.vocab_size, (MB_SIZE, SEQ_LEN), device=DEVICE, dtype=torch.long),
    "attention_mask": torch.ones((MB_SIZE, SEQ_LEN), device=DEVICE, dtype=torch.long),
}
print({k: v.shape for k,v in mb_kwargs.items()})

{'input_ids': torch.Size([1, 256]), 'attention_mask': torch.Size([1, 256])}


# GPU gc

In [ ]:
import gc, traceback
import torch
from torch.distributed.pipelining import pipeline, SplitPoint
from transformers import AutoModelForCausalLM

def hard_clear():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def fresh_root_gpu():
    base2 = AutoModelForCausalLM.from_pretrained(
        MODEL_ID_OR_PATH,
        torch_dtype=DTYPE,
        device_map="auto", # Changed from None to "auto"
    )
    # The .to(DEVICE) call might be redundant or problematic with device_map='auto'
    # as 'auto' already handles device placement. We will keep it for now and monitor.
    base2 = stabilize_for_capture(base2) # Removed .to(DEVICE) here to avoid conflicts with device_map='auto'
    return SimpleLlamaForPipeline(base2).to(DEVICE).eval()

@torch.no_grad()
def try_once_gpu(split_layer, sp, use_mask=True, seq_len=128):
    hard_clear()
    r = fresh_root_gpu()

    mb = {"input_ids": torch.randint(0, tokenizer.vocab_size, (MB_SIZE, seq_len), device=DEVICE, dtype=torch.long)}
    if use_mask:
        mb["attention_mask"] = torch.ones((MB_SIZE, seq_len), device=DEVICE, dtype=torch.long)

    split_spec = {f"model.layers.{split_layer}": sp}

    try:
        p = pipeline(module=r, mb_args=(), mb_kwargs=mb, split_spec=split_spec)
        ok = True
        out = p
    except Exception:
        ok = False
        out = "\n".join(traceback.format_exc().splitlines()[-35:])

    # 반드시 정리
    if ok:
        del p
    del r, mb
    hard_clear()
    return ok, (split_layer, sp, use_mask, seq_len), out

tests = []
for split_layer in [7]:  # ✅ 먼저 1개만 (시도 최소화)
    for sp in [SplitPoint.END, SplitPoint.BEGINNING]:
        for use_mask in [True, False]:
            for seq_len in [64, 128]:  # ✅ seq_len도 작게
                tests.append((split_layer, sp, use_mask, seq_len))

ok_case = None
for (sl, sp, um, L) in tests:
    ok, meta, out = try_once_gpu(sl, sp, um, L)
    if ok:
        ok_case = meta
        print("✅ FOUND OK:", meta)
        break
    else:
        print("❌", meta, "\n", out.splitlines()[-1])

print("best:", ok_case)

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]